# Getting access to Mat's Data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity-main")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity-main/

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity-main/import_substation.py

In [ ]:
#print names in each column of the info variable

print("Column names in info:",info.columns)

#residential column 6

# Trying geocode (geopy)

## list of substation names

In [ ]:
#print list of substation names

residential_names = info[info['Residential'] > 0.5]['Name']
print(residential_names)

# Trialing different mapping

## folium interative map

### initial mapping

In [ ]:
#seeing what's in the 'info' variable

info.head(3)

### fixing locations with missing lat or lan 

In [ ]:
# Filter rows where Latitude or Longitude is missing
missing_coords = info[info['Latitude'].isnull() | info['Longitude'].isnull()]

# Print all columns for those rows
print(missing_coords[['Name', 'Residential', 'Latitude', 'Longitude']])


In [ ]:
#adding coordinates for Dee Why West location

#lat: -33.7511
#lon: 151.2889

#update dataframe
info.loc[info['Name'] == 'Dee Why West', ['Latitude', 'Longitude']] = [-33.7511, 151.2889]

#verify update
print(info.loc[info['Name'] == 'Dee Why West', ['Name', 'Latitude', 'Longitude']])

In [ ]:
#create new map with all lon lat locations filled

import folium

# Use the full DataFrame now that lat/lon are filled
m = folium.Map(
    location=[info['Latitude'].mean(), info['Longitude'].mean()],
    zoom_start=10
)

# Add markers for every substation
for _, row in info.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=row['Name']
    ).add_to(m)

m

### colour code map by residential fraction

In [ ]:
import folium

m = folium.Map(
    location=[info['Latitude'].mean(), info['Longitude'].mean()],
    zoom_start=10
)

# Add color-coded markers
for _, row in info.iterrows():
    if row['Residential'] > 0.5:
        color = 'green'
    elif row['Residential'] > 0.2:
        color = 'orange'
    else:
        color = 'red'
    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"{row['Name']} (Residential fraction: {row['Residential']:.2f})"
    ).add_to(m)

# Add legend with HTML
legend_html = """
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 180px; height: 120px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.8;
     ">
&nbsp;<b>Residential Fraction</b><br>
&nbsp;<i style="color:green;">●</i> > 0.5<br>
&nbsp;<i style="color:orange;">●</i> 0.2 – 0.5<br>
&nbsp;<i style="color:red;">●</i> < 0.2
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m


In [ ]:
m.fit_bounds(m.get_bounds())   # ensures all substations are visible
m.save("substations_map.html")